In [1]:
import pandas as pd

path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\03_advanced_prep\lc_after_03_advanced_prep_basic+test_20260108_1516.csv"

df = pd.read_csv(path, low_memory=False)

print("Loaded:", df.shape)
df.head()

Loaded: (221428, 108)


,percent_bc_gt_75,num_tl_op_past_12m,zip_code,revol_bal,last_fico_range_high,sub_grade,emp_title,last_fico_range_low,total_rev_hi_lim,term,...,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Source Verified,verification_status_Verified
0,NaN,NaN,860xx,9.521422,749.0,B2,NaN,745.0,NaN,36,...,0,0,0,0,0,0,0,1,0,1
1,NaN,NaN,309xx,7.431300,499.0,C4,Ryder,0.0,NaN,60,...,0,0,0,0,0,0,0,1,1,0
2,NaN,NaN,606xx,7.991931,739.0,C5,NaN,735.0,NaN,36,...,0,1,0,0,0,0,0,1,0,0
3,NaN,NaN,917xx,8.630343,604.0,C1,AIR RESOURCES BOARD,600.0,NaN,36,...,0,0,0,0,0,0,0,1,1,0
4,NaN,NaN,972xx,10.232216,684.0,B5,University Medical Group,680.0,NaN,60,...,0,0,0,0,0,0,0,1,1,0


In [3]:
import pandas as pd
import numpy as np

path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\03_advanced_prep\lc_after_03_advanced_prep_basic+test_20260108_1516.csv"
df = pd.read_csv(path, low_memory=False)

print("Loaded:", df.shape)
print(df.columns.tolist()[:40])
df.head()


Loaded: (221428, 108)
['percent_bc_gt_75', 'num_tl_op_past_12m', 'zip_code', 'revol_bal', 'last_fico_range_high', 'sub_grade', 'emp_title', 'last_fico_range_low', 'total_rev_hi_lim', 'term', 'title', 'avg_cur_bal', 'int_rate', 'dti', 'total_bc_limit', 'emp_length', 'mo_sin_rcnt_tl', 'mo_sin_old_rev_tl_op', 'mths_since_recent_bc', 'revol_util', 'annual_inc', 'num_rev_tl_bal_gt_0', 'acc_open_past_24mths', 'mths_since_recent_inq', 'grade', 'bc_util', 'inq_last_6mths', 'issue_d', 'is_default', 'fico_range_low', 'mo_sin_rcnt_rev_tl_op', 'bc_open_to_buy', 'tot_hi_cred_lim', 'pct_tl_nvr_dlq', 'loan_amnt', 'desc', 'issue_month_start', 'issue_ym', 'month_idx', 'funded_ratio']


,percent_bc_gt_75,num_tl_op_past_12m,zip_code,revol_bal,last_fico_range_high,sub_grade,emp_title,last_fico_range_low,total_rev_hi_lim,term,...,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Source Verified,verification_status_Verified
0,NaN,NaN,860xx,9.521422,749.0,B2,NaN,745.0,NaN,36,...,0,0,0,0,0,0,0,1,0,1
1,NaN,NaN,309xx,7.431300,499.0,C4,Ryder,0.0,NaN,60,...,0,0,0,0,0,0,0,1,1,0
2,NaN,NaN,606xx,7.991931,739.0,C5,NaN,735.0,NaN,36,...,0,1,0,0,0,0,0,1,0,0
3,NaN,NaN,917xx,8.630343,604.0,C1,AIR RESOURCES BOARD,600.0,NaN,36,...,0,0,0,0,0,0,0,1,1,0
4,NaN,NaN,972xx,10.232216,684.0,B5,University Medical Group,680.0,NaN,60,...,0,0,0,0,0,0,0,1,1,0


In [4]:
# ---- parameters ----
TEXT_COL = "desc"          # שנה אם צריך
TIME_COL = "issue_ym"      # שנה אם אצלך נקרא אחרת
TARGET_COL = "is_default"  # 0/1

# ---- zip3 creation ----
def to_zip3(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    # מקרים נפוצים בלנדינגקלאב: "123xx", "123XX", "123"
    s = s.str.replace(r"[^0-9xX]", "", regex=True).str.lower()
    z3 = s.str.extract(r"^(\d{3})")[0]
    return z3

if "zip3" not in df.columns:
    if "zip_code" in df.columns:
        df["zip3"] = to_zip3(df["zip_code"])
    elif "addr_state" in df.columns:
        # fallback חלש – לא מומלץ, עדיף zip_code
        df["zip3"] = np.nan
    else:
        df["zip3"] = np.nan

# ---- enforce types ----
df[TEXT_COL] = df.get(TEXT_COL, "").fillna("").astype(str)
df["zip3"] = df["zip3"].astype("string")

# זמן
df[TIME_COL] = df[TIME_COL].astype(str).str.strip()
# יעד
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0).astype(int)

# סינון מינימלי
df = df[df["zip3"].notna() & (df["zip3"].str.len() == 3)]
df = df[df[TIME_COL].notna() & (df[TIME_COL] != "nan")]

print("After basic cleaning:", df.shape)
df[[TIME_COL, "zip3", TARGET_COL, TEXT_COL]].head()


After basic cleaning: (201254, 108)


,issue_ym,zip3,is_default,desc
0,2011-12,860,0,Borrower added on 12/22/11 > I need to upgra...
1,2011-12,309,1,Borrower added on 12/22/11 > I plan to use t...
2,2011-12,606,0,
3,2011-12,917,0,Borrower added on 12/21/11 > to pay for prop...
4,2011-12,972,0,Borrower added on 12/21/11 > I plan on combi...


פיצ’רי טקסט “סגנון/מורכבות” (בלי מילונים)

In [5]:
import re

def text_features(s: pd.Series) -> pd.DataFrame:
    x = s.fillna("").astype(str)

    n_chars = x.str.len()

    # word count
    n_words = x.str.findall(r"\S+").str.len()

    # sentence-ish count (fallback)
    n_sents = x.str.count(r"[.!?]+").clip(lower=1)

    # avg word length (על בסיס אותיות/ספרות, לא רווחים)
    stripped = x.str.replace(r"\s+", "", regex=True)
    avg_word_len = (stripped.str.len() / n_words.replace(0, np.nan)).fillna(0)

    # upper ratio (בעיקר לאנגלית)
    n_upper = x.str.count(r"[A-Z]")
    n_alpha = x.str.count(r"[A-Za-z]")
    upper_ratio = (n_upper / n_alpha.replace(0, np.nan)).fillna(0)

    # digits ratio
    digit_ratio = (x.str.count(r"[0-9]") / n_chars.replace(0, np.nan)).fillna(0)

    # punctuation emphasis
    exclam = x.str.count(r"!")
    quest  = x.str.count(r"\?")
    
    # word length SD + TTR (lexical diversity)
    def _wlen_sd_and_ttr(t: str):
        toks = re.split(r"\s+", t.lower().strip())
        toks = [re.sub(r"[^a-z]", "", w) for w in toks]  # שמרני: אנגלית
        toks = [w for w in toks if w]
        if len(toks) == 0:
            return 0.0, 0.0
        wlens = [len(w) for w in toks]
        wlen_sd = float(np.std(wlens, ddof=1)) if len(wlens) > 1 else 0.0
        ttr = float(len(set(toks)) / len(toks))
        return wlen_sd, ttr

    tmp = x.apply(_wlen_sd_and_ttr)
    wlen_sd = tmp.apply(lambda z: z[0])
    ttr     = tmp.apply(lambda z: z[1])

    return pd.DataFrame({
        "txt_chars": n_chars,
        "txt_words": n_words,
        "txt_sents": n_sents,
        "txt_avg_word_len": avg_word_len,
        "txt_upper_ratio": upper_ratio,
        "txt_digit_ratio": digit_ratio,
        "txt_exclam": exclam,
        "txt_question": quest,
        "txt_wordlen_sd": wlen_sd,
        "txt_ttr": ttr
    })

feat_basic = text_features(df[TEXT_COL])
df_feat = pd.concat([df.reset_index(drop=True), feat_basic.reset_index(drop=True)], axis=1)

df_feat[["txt_words","txt_ttr","txt_upper_ratio","txt_digit_ratio"]].describe().T


,count,mean,std,min,25%,50%,75%,max
txt_words,201254.0,19.487101,37.661044,0.0,0.0,0.0,27.000000,815.000000
txt_ttr,201254.0,0.393767,0.438178,0.0,0.0,0.0,0.866667,1.000000
txt_upper_ratio,201254.0,0.021154,0.065367,0.0,0.0,0.0,0.028571,1.000000
txt_digit_ratio,201254.0,0.024798,0.034005,0.0,0.0,0.0,0.043165,0.368794


Topic proportions (LDA) עם scikit-learn

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# פרמטרים שמרניים כדי לא להעמיס זיכרון
N_TOPICS = 8
MAX_FEATURES = 20000
MIN_DF = 20      # תתאים לפי גודל דאטה
MAX_DF = 0.95

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES
)

X = vectorizer.fit_transform(df_feat[TEXT_COL].values)

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=123,
    learning_method="batch",
    max_iter=15
)

theta = lda.fit_transform(X)  # n x K topic proportions

topic_cols = [f"topic_{i+1}" for i in range(N_TOPICS)]
df_topics = pd.DataFrame(theta, columns=topic_cols)

df_all = pd.concat([df_feat.reset_index(drop=True), df_topics.reset_index(drop=True)], axis=1)

print("Final shape:", df_all.shape)
df_all[topic_cols].head()


Final shape: (201254, 126)


,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8
0,0.020840,0.020849,0.020866,0.020848,0.221104,0.653786,0.020845,0.020862
1,0.002662,0.609624,0.002662,0.002664,0.374394,0.002665,0.002665,0.002664
2,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000
3,0.006951,0.180753,0.006958,0.153109,0.287600,0.350717,0.006953,0.006960
4,0.006261,0.441983,0.006254,0.006259,0.006256,0.006255,0.006254,0.520478


In [11]:
topic_cols = [c for c in df_all.columns if c.startswith("topic_")]
uniform = (df_all[topic_cols].round(6).nunique(axis=1) == 1) & (df_all[topic_cols].round(6).iloc[:,0] == round(1/len(topic_cols),6))
print("Uniform rows:", uniform.mean(), "share")
df_all.loc[uniform, ["desc","txt_words"]].head(10)


Uniform rows: 0.5413258866904509 share


,desc,txt_words
2,,0
5,,0
11,,0
14,,0
15,,0
16,,0
18,,0
22,,0
23,,0
24,,0


In [12]:
import numpy as np
import pandas as pd

TEXT_COL = "desc"
TARGET_COL = "is_default"

df_all["has_text"] = (df_all[TEXT_COL].fillna("").str.strip().str.len() > 0).astype(int)

# 1) האם יש הבדל בשיעור טקסט בין ZIP3?
has_text_by_zip3 = (
    df_all.groupby("zip3")
    .agg(n=("zip3","size"),
         share_has_text=("has_text","mean"),
         default_rate=(TARGET_COL,"mean"))
    .sort_values("n", ascending=False)
)

has_text_by_zip3.head(30)

# 2) השוואה כללית: טקסט כן/לא
df_all.groupby("has_text")[TARGET_COL].agg(["count","mean"])


,count,mean
has_text,,
0,108944,0.157072
1,92310,0.153407


In [13]:
MIN_N = 500

zip_stats = (
    df_all.groupby("zip3")
    .agg(
        n=("zip3","size"),
        share_has_text=("has_text","mean"),
        default_rate=("is_default","mean")
    )
    .query("n >= @MIN_N")
    .sort_values("share_has_text", ascending=False)
)

zip_stats.head(20), zip_stats.tail(20)


(         n  share_has_text  default_rate
 zip3                                    
 840    854        0.508197      0.162763
 551    552        0.507246      0.146739
 801    653        0.503828      0.102603
 761    612        0.503268      0.178105
 981    783        0.499361      0.114943
 920   1144        0.499126      0.147727
 333    886        0.496614      0.191874
 337    633        0.492891      0.161137
 282    700        0.491429      0.158571
 334   1203        0.491272      0.165420
 802   1004        0.491036      0.114542
 481   1031        0.489816      0.169738
 335    652        0.489264      0.180982
 480    809        0.488257      0.165637
 432    595        0.487395      0.149580
 342    521        0.485605      0.157390
 775    705        0.485106      0.163121
 941   1305        0.484291      0.096552
 752    822        0.484185      0.125304
 322    649        0.483821      0.160247,
          n  share_has_text  default_rate
 zip3                            

In [14]:
zip_stats = zip_stats.copy()

corr = zip_stats["share_has_text"].corr(zip_stats["default_rate"])
print("Corr(share_has_text, default_rate) across ZIP3:", corr)

# אופציונלי: Spearman בלי ספריות נוספות
rank_corr = zip_stats["share_has_text"].rank().corr(zip_stats["default_rate"].rank())
print("Spearman-ish:", rank_corr)


Corr(share_has_text, default_rate) across ZIP3: -0.2410430252743741
Spearman-ish: -0.22340839588546013


In [16]:
import numpy as np
import pandas as pd

# עמודות שאסור להכניס (דליפה/מזהים/יעד/זמן/קבוצה/טקסט)
ban = {
    "is_default", "loan_status", "default", "target",
    "zip3", "zip_code", "addr_state",
    "issue_ym", "issue_d", "_time", "_month",
    "desc", "title", "emp_title", "url", "member_id", "id"
}

# בוחר רק מספריים
num_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
BASE_FEATURES = [c for c in num_cols if c not in ban]

print("Num candidates:", len(num_cols))
print("BASE_FEATURES:", len(BASE_FEATURES))
print(BASE_FEATURES[:30])


Num candidates: 117
BASE_FEATURES: 116
['percent_bc_gt_75', 'num_tl_op_past_12m', 'revol_bal', 'last_fico_range_high', 'last_fico_range_low', 'total_rev_hi_lim', 'term', 'avg_cur_bal', 'int_rate', 'dti', 'total_bc_limit', 'emp_length', 'mo_sin_rcnt_tl', 'mo_sin_old_rev_tl_op', 'mths_since_recent_bc', 'revol_util', 'annual_inc', 'num_rev_tl_bal_gt_0', 'acc_open_past_24mths', 'mths_since_recent_inq', 'bc_util', 'inq_last_6mths', 'fico_range_low', 'mo_sin_rcnt_rev_tl_op', 'bc_open_to_buy', 'tot_hi_cred_lim', 'pct_tl_nvr_dlq', 'loan_amnt', 'month_idx', 'funded_ratio']


In [17]:
# נסה להמיר עמודות object למספרים, בלי להרוס קטגוריות מובהקות
obj_cols = df_all.select_dtypes(include=["object", "string"]).columns.tolist()

for c in obj_cols:
    if c in ban:
        continue
    # המרה מספרית אם אפשר (אחרת ייצא NaN)
    converted = pd.to_numeric(df_all[c], errors="coerce")
    # אם לפחות 70% לא-NaN, נשתמש בזה כמספרי
    if converted.notna().mean() >= 0.7:
        df_all[c] = converted

num_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
BASE_FEATURES = [c for c in num_cols if c not in ban]

print("After coercion, BASE_FEATURES:", len(BASE_FEATURES))
print(BASE_FEATURES[:30])


After coercion, BASE_FEATURES: 116
['percent_bc_gt_75', 'num_tl_op_past_12m', 'revol_bal', 'last_fico_range_high', 'last_fico_range_low', 'total_rev_hi_lim', 'term', 'avg_cur_bal', 'int_rate', 'dti', 'total_bc_limit', 'emp_length', 'mo_sin_rcnt_tl', 'mo_sin_old_rev_tl_op', 'mths_since_recent_bc', 'revol_util', 'annual_inc', 'num_rev_tl_bal_gt_0', 'acc_open_past_24mths', 'mths_since_recent_inq', 'bc_util', 'inq_last_6mths', 'fico_range_low', 'mo_sin_rcnt_rev_tl_op', 'bc_open_to_buy', 'tot_hi_cred_lim', 'pct_tl_nvr_dlq', 'loan_amnt', 'month_idx', 'funded_ratio']


In [18]:
# מוריד עמודות עם יותר מדי NA או שונות 0
keep = []
for c in BASE_FEATURES:
    s = df_all[c]
    if s.notna().mean() < 0.8:
        continue
    if s.nunique(dropna=True) <= 1:
        continue
    keep.append(c)

BASE_FEATURES = keep
print("Filtered BASE_FEATURES:", len(BASE_FEATURES))


Filtered BASE_FEATURES: 104


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

TIME_COL = "issue_ym"
TARGET_COL = "is_default"
GROUP_COL = "zip3"
TEXT_COL = "desc"

# has_text
df_all["has_text"] = (df_all[TEXT_COL].fillna("").str.strip().str.len() > 0).astype(int)

# time split
df_all["_time"] = pd.to_datetime(df_all[TIME_COL] + "-01", errors="coerce")
df_all = df_all[df_all["_time"].notna()].sort_values("_time")
df_all["_month"] = df_all["_time"].dt.to_period("M").astype(str)

months = df_all["_month"].drop_duplicates().sort_values()
cut = int(len(months) * 0.8)
train_months = set(months.iloc[:cut])
test_months  = set(months.iloc[cut:])

train = df_all[df_all["_month"].isin(train_months)].copy()
test  = df_all[df_all["_month"].isin(test_months)].copy()

def fit_predict(train_df, test_df, features):
    Xtr = train_df[features].fillna(0)
    ytr = train_df[TARGET_COL].astype(int).values
    Xte = test_df[features].fillna(0)
    yte = test_df[TARGET_COL].astype(int).values

    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(max_iter=2000))
    ])
    pipe.fit(Xtr, ytr)
    p = pipe.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, p)
    return p, auc

def fair_table_by_group(df, group_col=GROUP_COL, y_col=TARGET_COL, yhat_col="pred_y",
                        min_n=300, min_pos=50, min_neg=50):
    rows = []
    for g, d in df.groupby(group_col):
        n = len(d)
        y = d[y_col].values
        yhat = d[yhat_col].values

        pos = int((y == 1).sum())
        neg = int((y == 0).sum())
        if (n < min_n) or (pos < min_pos) or (neg < min_neg):
            continue

        tp = int(((yhat==1) & (y==1)).sum())
        fp = int(((yhat==1) & (y==0)).sum())
        tn = int(((yhat==0) & (y==0)).sum())
        fn = int(((yhat==0) & (y==1)).sum())

        tpr = tp / pos
        fpr = fp / neg
        fnr = fn / pos
        ppv = tp / max(1, int((yhat==1).sum()))

        rows.append([g, n, pos, neg, y.mean(), tpr, fpr, fnr, ppv])

    return (pd.DataFrame(rows, columns=[group_col,"n","pos","neg","base_rate","TPR","FPR","FNR","PPV"])
              .sort_values("n", ascending=False))

# M0
p0, auc0 = fit_predict(train, test, BASE_FEATURES)
test0 = test.copy()
test0["pred_p"] = p0
test0["pred_y"] = (p0 >= 0.5).astype(int)
fair0 = fair_table_by_group(test0)

# M1 (+has_text)
p1, auc1 = fit_predict(train, test, BASE_FEATURES + ["has_text"])
test1 = test.copy()
test1["pred_p"] = p1
test1["pred_y"] = (p1 >= 0.5).astype(int)
fair1 = fair_table_by_group(test1)

print("AUC M0:", round(auc0,4))
print("AUC M1:", round(auc1,4))

cmp = fair0.merge(fair1, on=GROUP_COL, suffixes=("_M0","_M1"))
cmp["FPR_diff"] = cmp["FPR_M1"] - cmp["FPR_M0"]
cmp["TPR_diff"] = cmp["TPR_M1"] - cmp["TPR_M0"]

cmp.sort_values("n_M0", ascending=False).head(25)[
    [GROUP_COL, "n_M0", "base_rate_M0", "TPR_M0","TPR_M1","TPR_diff","FPR_M0","FPR_M1","FPR_diff"]
]


AUC M0: 0.9019
AUC M1: 0.9018


,zip3,n_M0,base_rate_M0,TPR_M0,TPR_M1,TPR_diff,FPR_M0,FPR_M1,FPR_diff
0,945,1454,0.126547,0.434783,0.434783,0.000000,0.056693,0.056693,0.000000
1,750,1298,0.153313,0.361809,0.361809,0.000000,0.030937,0.031847,0.000910
2,112,1266,0.183254,0.344828,0.344828,0.000000,0.067698,0.067698,0.000000
3,606,1152,0.119792,0.362319,0.362319,0.000000,0.029586,0.029586,0.000000
4,300,1051,0.136061,0.496503,0.496503,0.000000,0.052863,0.052863,0.000000
5,900,1045,0.157895,0.466667,0.466667,0.000000,0.036364,0.036364,0.000000
6,100,1029,0.127308,0.366412,0.366412,0.000000,0.041203,0.042316,0.001114
7,331,997,0.192578,0.510417,0.505208,-0.005208,0.093168,0.093168,0.000000
8,917,950,0.158947,0.470199,0.470199,0.000000,0.061327,0.060075,-0.001252
9,891,892,0.209641,0.513369,0.508021,-0.005348,0.080851,0.080851,0.000000


In [20]:
df_all[df_all["has_text"] == 1]


,percent_bc_gt_75,num_tl_op_past_12m,zip_code,revol_bal,last_fico_range_high,sub_grade,emp_title,last_fico_range_low,total_rev_hi_lim,term,...,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,has_text,_time,_month
29221,NaN,NaN,857xx,6.318968,819.0,B3,GRG Construction,815.0,NaN,36,...,0.003685,0.003685,0.003684,0.232504,0.083404,0.665674,0.003682,1,2010-01-01,2010-01
29609,NaN,NaN,980xx,11.302501,564.0,D1,NaN,560.0,NaN,36,...,0.012523,0.395516,0.012510,0.529403,0.012515,0.012516,0.012513,1,2010-01-01,2010-01
29608,NaN,NaN,401xx,7.354362,529.0,C5,fiddlers llc,525.0,NaN,36,...,0.217224,0.004468,0.004469,0.004475,0.504475,0.004471,0.004480,1,2010-01-01,2010-01
29607,NaN,NaN,113xx,6.093570,504.0,B4,empire merchants,500.0,NaN,36,...,0.002913,0.002911,0.002911,0.431783,0.002909,0.240086,0.313577,1,2010-01-01,2010-01
29606,NaN,NaN,309xx,9.181529,764.0,C1,Computers Universal Inc,760.0,NaN,36,...,0.262143,0.001034,0.001035,0.619252,0.001034,0.058587,0.055881,1,2010-01-01,2010-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29710,100.0,1.0,908xx,8.364508,679.0,B3,Truck Driver,675.0,8.594339,36,...,0.011367,0.783390,0.011374,0.011365,0.011367,0.148392,0.011373,1,2013-12-01,2013-12
29719,NaN,2.0,125xx,8.743213,499.0,D3,Inventory control Manager,0.0,9.200391,36,...,0.058065,0.005448,0.005443,0.005440,0.005439,0.005439,0.114385,1,2013-12-01,2013-12
29714,50.0,1.0,233xx,9.386560,724.0,A5,Registered Nurse,720.0,9.635673,36,...,0.005214,0.005220,0.613858,0.005213,0.005214,0.354845,0.005220,1,2013-12-01,2013-12
29716,100.0,1.0,274xx,8.352790,684.0,C3,A/R Specialist,680.0,8.556606,36,...,0.447818,0.006957,0.308372,0.006963,0.006962,0.209010,0.006960,1,2013-12-01,2013-12


In [21]:
print(df_all["has_text"].mean())
print(df_all[df_all["has_text"] == 1].shape)


0.4586741133095491
(92310, 129)


In [22]:
import re

def clean_desc(s: pd.Series) -> pd.Series:
    x = s.fillna("").astype(str)

    # 1) remove HTML tags
    x = x.str.replace(r"<[^>]+>", " ", regex=True)

    # 2) remove the standard LC prefix "Borrower added on 12/22/11 >"
    # handles variations in spacing and the ">" separator
    x = x.str.replace(
        r"^\s*borrower\s+added\s+on\s+\d{1,2}/\d{1,2}/\d{2,4}\s*(?:>\s*)?",
        "",
        regex=True,
        flags=re.IGNORECASE
    )

    # 3) normalize whitespace
    x = x.str.replace(r"\s+", " ", regex=True).str.strip()

    return x

df_all["desc_clean"] = clean_desc(df_all["desc"])


In [23]:
df_txt = df_all[df_all["has_text"] == 1].copy()

X = vectorizer.fit_transform(df_txt["desc_clean"].astype(str).values)


In [24]:
df_all.loc[df_all["has_text"]==1, ["desc","desc_clean"]].head(5)


,desc,desc_clean
29221,Borrower added on 01/23/10 > Interest rates ...,Interest rates on my credit cards have gone up...
29609,Borrower added on 12/18/09 > Business workin...,Business working capital Borrower added on 12/...
29608,Borrower added on 12/19/09 > i plan to use t...,i plan to use the funds to do a little home im...
29607,Borrower added on 12/19/09 > I have a great ...,I have a great job and get paid well. I wanted...
29606,Borrower added on 12/21/09 > I'll be moving ...,I'll be moving to another country for work as ...


In [25]:
share_empty_after = (df_all["desc_clean"].str.len() == 0).mean()
print("Empty after cleaning:", share_empty_after)


Empty after cleaning: 0.5413308555357906


In [26]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# ======================
# Parameters
# ======================
TEXT_COL = "desc_clean"
N_TOPICS = 10          # 8-12 זה טווח טוב להתחלה
MIN_DF = 10            # בגלל 90K+ טקסטים אפשר 10-20
MAX_DF = 0.95
MAX_FEATURES = 30000
RANDOM_STATE = 123

# ======================
# Filter to non-empty text
# ======================
df_all["has_text_clean"] = (df_all[TEXT_COL].fillna("").str.strip().str.len() > 0).astype(int)
df_txt = df_all[df_all["has_text_clean"] == 1].copy()

print("Rows with non-empty desc_clean:", df_txt.shape)

# ======================
# Vectorize (bag of words)
# ======================
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES
)

X = vectorizer.fit_transform(df_txt[TEXT_COL].astype(str).values)
print("DTM shape:", X.shape, "| nnz:", X.nnz)

# ======================
# Fit LDA
# ======================
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=15
)

theta = lda.fit_transform(X)  # (n_docs, K)
topic_cols = [f"topic_{i+1}" for i in range(N_TOPICS)]

df_txt_topics = pd.DataFrame(theta, index=df_txt.index, columns=topic_cols)

# ======================
# Merge topics back into df_all (NaN for empty texts)
# ======================
for c in topic_cols:
    df_all[c] = np.nan

df_all.loc[df_txt_topics.index, topic_cols] = df_txt_topics

print("Topics added. Example:")
print(df_all.loc[df_txt_topics.index[:5], topic_cols].head())

# ======================
# Show top words per topic (sanity check)
# ======================
feature_names = np.array(vectorizer.get_feature_names_out())
TOP_WORDS = 12

for k in range(N_TOPICS):
    top_idx = np.argsort(lda.components_[k])[::-1][:TOP_WORDS]
    top_terms = feature_names[top_idx]
    print(f"\nTopic {k+1}: " + ", ".join(top_terms))


Rows with non-empty desc_clean: (92309, 131)
DTM shape: (92309, 4836) | nnz: 1286307
Topics added. Example:
        topic_1   topic_2   topic_3   topic_4   topic_5   topic_6   topic_7  \
29221  0.003126  0.480259  0.003126  0.122584  0.003127  0.234029  0.003125   
29609  0.012503  0.012501  0.012503  0.012504  0.012504  0.636494  0.012500   
29608  0.003847  0.003848  0.003847  0.003847  0.519593  0.003847  0.003847   
29607  0.002326  0.332878  0.601241  0.002326  0.049597  0.002326  0.002326   
29606  0.103485  0.079175  0.252799  0.000788  0.214018  0.334424  0.000788   

        topic_8   topic_9  topic_10  
29221  0.003126  0.055630  0.091868  
29609  0.012500  0.012501  0.263490  
29608  0.003846  0.449632  0.003847  
29607  0.002327  0.002326  0.002326  
29606  0.000788  0.012949  0.000788  

Topic 1: bills, pay, medical, added, borrower, loan, expenses, need, car, help, family, money

Topic 2: debt, pay, credit, just, help, make, paying, want, payment, cards, payments, like

T

sanity check

In [27]:
feature_names = np.array(vectorizer.get_feature_names_out())

for k in range(N_TOPICS):
    top_idx = np.argsort(lda.components_[k])[::-1][:12]
    print(f"\nTopic {k+1}:",
          ", ".join(feature_names[top_idx]))



Topic 1: bills, pay, medical, added, borrower, loan, expenses, need, car, help, family, money

Topic 2: debt, pay, credit, just, help, make, paying, want, payment, cards, payments, like

Topic 3: loan, thank, borrower, added, lending, club, thanks, help, quot, consideration, time, investors

Topic 4: credit, cards, pay, loan, high, added, card, borrower, consolidate, want, paying, use

Topic 5: years, job, credit, loan, stable, time, borrower, pay, good, plan, added, company

Topic 6: business, loan, new, year, school, expenses, years, income, used, company, work, moving

Topic 7: payment, monthly, loan, payments, month, consolidate, pay, bills, loans, make, lower, debts

Topic 8: credit, card, debt, rate, loan, pay, lower, consolidate, high, cards, higher, rates

Topic 9: home, house, new, loan, need, improvement, improvements, purchase, repairs, kitchen, added, borrower

Topic 10: debt, consolidation, free, loan, added, borrower, consolidate, goal, years, help, consolidating, want


In [28]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation

TEXT_COL = "desc_clean"
N_TOPICS = 10
MIN_DF = 10
MAX_DF = 0.95
MAX_FEATURES = 30000
RANDOM_STATE = 123

# 1) filter to non-empty cleaned text
df_all["has_text_clean"] = (df_all[TEXT_COL].fillna("").str.strip().str.len() > 0).astype(int)
df_txt = df_all[df_all["has_text_clean"] == 1].copy()

# 2) custom stopwords (add what you saw contaminating topics)
custom_stop = {
    "added","borrower","loan","lending","club",
    "thanks","thank","quot","consideration","time",
    "just","like","want","make","need"
}

stop_words = set(ENGLISH_STOP_WORDS).union(custom_stop)

# 3) vectorize
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=list(stop_words),
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES
)

X = vectorizer.fit_transform(df_txt[TEXT_COL].astype(str).values)
print("DTM shape:", X.shape, "| nnz:", X.nnz)

# 4) LDA
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=15
)

theta = lda.fit_transform(X)
topic_cols = [f"topic_{i+1}" for i in range(N_TOPICS)]
df_txt[topic_cols] = theta

# 5) merge back (NaN for empty texts)
for c in topic_cols:
    df_all[c] = np.nan
df_all.loc[df_txt.index, topic_cols] = df_txt[topic_cols]

# 6) print top words per topic (sanity check)
feature_names = np.array(vectorizer.get_feature_names_out())
TOP_WORDS = 12

for k in range(N_TOPICS):
    top_idx = np.argsort(lda.components_[k])[::-1][:TOP_WORDS]
    print(f"\nTopic {k+1}: " + ", ".join(feature_names[top_idx]))


DTM shape: (92309, 4821) | nnz: 1138083

Topic 1: business, purchase, used, vehicle, new, small, truck, car, buy, start, work, equipment

Topic 2: help, pay, credit, debt, money, start, college, able, school, life, paying, years

Topic 3: home, house, improvement, improvements, purchase, repairs, buy, pool, used, finish, vacation, family

Topic 4: job, years, stable, pay, plan, good, monthly, payment, credit, payments, use, funds

Topic 5: debt, consolidation, free, years, consolidate, card, credit, goal, help, financial, looking, plan

Topic 6: years, year, income, paid, credit, property, questions, company, currently, cash, current, investment

Topic 7: bills, pay, medical, car, expenses, money, help, wedding, year, little, paying, getting

Topic 8: payment, credit, consolidate, pay, monthly, debt, cards, payments, rate, lower, card, month

Topic 9: credit, pay, cards, card, high, debt, rate, consolidate, higher, paying, payoff, rates

Topic 10: new, house, home, roof, replace, movin

1.Business / Vehicle / Equipment
2.Debt / College / Life expenses
3.Home improvement
4.Stable job / monthly payment
5.Debt consolidation
6.Income / property
7.Medical / wedding / bills
8.Monthly payment / lower rate
9.High rate / payoff
10.Home repair / moving

topics by ZIP3

In [29]:
topic_means_by_zip = (
    df_txt.groupby("zip3")[topic_cols]
    .mean()
)

topic_variation = topic_means_by_zip.std().sort_values(ascending=False)
print(topic_variation)


topic_9     0.067011
topic_8     0.057995
topic_5     0.041584
topic_4     0.038814
topic_2     0.037079
topic_7     0.037069
topic_3     0.029324
topic_1     0.027350
topic_10    0.026523
topic_6     0.025458
dtype: float64


הגדרת BASE_FEATURES

In [30]:
import numpy as np

# --- אל תכניס ---
BAN = {
    "is_default","desc","desc_clean","title","emp_title",
    "zip_code","zip3","issue_d","issue_month_start","issue_ym","month_idx",
}

# מועמדים טובים מתוך הרשימה שלך
candidate = [
    "loan_amnt","funded_ratio","term","int_rate","installment",
    "annual_inc","dti",
    "fico_range_low","last_fico_range_low","last_fico_range_high",
    "revol_bal","revol_util",
    "total_rev_hi_lim","tot_hi_cred_lim",
    "avg_cur_bal",
    "inq_last_6mths","mths_since_recent_inq",
    "acc_open_past_24mths","num_rev_tl_bal_gt_0","num_tl_op_past_12m",
    "pct_tl_nvr_dlq",
    "mo_sin_rcnt_tl","mo_sin_rcnt_rev_tl_op","mo_sin_old_rev_tl_op",
    "bc_util","total_bc_limit","total_bc_limit","bc_open_to_buy",
    "total_bc_limit","mths_since_recent_bc","percent_bc_gt_75"
]

# הוספת one-hot שכבר קיימים אצלך (purpose / home_ownership / verification / addr_state)
onehot_prefixes = (
    "addr_state_",
    "purpose_",
    "home_ownership_",
    "verification_status_"
)

onehot_cols = [c for c in df_all.columns if c.startswith(onehot_prefixes)]

BASE_FEATURES = [c for c in candidate if c in df_all.columns and c not in BAN] + onehot_cols

# מסנן כפילויות ושדות קבועים/עם הרבה NA
BASE_FEATURES = list(dict.fromkeys(BASE_FEATURES))
BASE_FEATURES = [
    c for c in BASE_FEATURES
    if c in df_all.columns
    and df_all[c].notna().mean() >= 0.70
    and df_all[c].nunique(dropna=True) > 1
    and c not in BAN
]

print("BASE_FEATURES:", len(BASE_FEATURES))
print(BASE_FEATURES[:40])


BASE_FEATURES: 93
['loan_amnt', 'funded_ratio', 'term', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'last_fico_range_low', 'last_fico_range_high', 'revol_bal', 'revol_util', 'total_rev_hi_lim', 'tot_hi_cred_lim', 'avg_cur_bal', 'inq_last_6mths', 'mths_since_recent_inq', 'acc_open_past_24mths', 'num_rev_tl_bal_gt_0', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'mo_sin_rcnt_tl', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_old_rev_tl_op', 'bc_util', 'total_bc_limit', 'bc_open_to_buy', 'mths_since_recent_bc', 'percent_bc_gt_75', 'addr_state_AL', 'addr_state_AR', 'addr_state_AZ', 'addr_state_CA', 'addr_state_CO', 'addr_state_CT', 'addr_state_DC', 'addr_state_DE', 'addr_state_FL', 'addr_state_GA', 'addr_state_HI', 'addr_state_IA']


In [31]:
TEXT_COL = "desc_clean"
df_all["has_text_clean"] = (df_all[TEXT_COL].fillna("").str.strip().str.len() > 0).astype(int)
df_txt = df_all[df_all["has_text_clean"] == 1].copy()

print("df_txt:", df_txt.shape)


df_txt: (92309, 133)


In [32]:
import pandas as pd

TIME_COL = "issue_ym"
TARGET_COL = "is_default"
GROUP_COL = "zip3"

df_txt["_time"] = pd.to_datetime(df_txt[TIME_COL] + "-01", errors="coerce")
df_txt = df_txt[df_txt["_time"].notna()].sort_values("_time")
df_txt["_month"] = df_txt["_time"].dt.to_period("M").astype(str)

months = df_txt["_month"].drop_duplicates().sort_values()
cut = int(len(months) * 0.8)
train_months = set(months.iloc[:cut])
test_months  = set(months.iloc[cut:])

train = df_txt[df_txt["_month"].isin(train_months)].copy()
test  = df_txt[df_txt["_month"].isin(test_months)].copy()

print("Train:", train.shape, "| Test:", test.shape)


Train: (55236, 133) | Test: (37073, 133)


In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

def fit_predict(train_df, test_df, features):
    Xtr = train_df[features].fillna(0)
    ytr = train_df[TARGET_COL].astype(int).values
    Xte = test_df[features].fillna(0)
    yte = test_df[TARGET_COL].astype(int).values

    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(max_iter=2000))
    ])
    pipe.fit(Xtr, ytr)
    p = pipe.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, p)
    return p, auc

def fair_table_by_group(df, group_col=GROUP_COL, y_col=TARGET_COL, yhat_col="pred_y",
                        min_n=300, min_pos=50, min_neg=50):
    rows = []
    for g, d in df.groupby(group_col):
        n = len(d)
        y = d[y_col].values
        yhat = d[yhat_col].values

        pos = int((y == 1).sum())
        neg = int((y == 0).sum())
        if (n < min_n) or (pos < min_pos) or (neg < min_neg):
            continue

        tp = int(((yhat==1) & (y==1)).sum())
        fp = int(((yhat==1) & (y==0)).sum())
        fn = int(((yhat==0) & (y==1)).sum())

        tpr = tp / pos
        fpr = fp / neg
        fnr = fn / pos
        ppv = tp / max(1, int((yhat==1).sum()))

        rows.append([g, n, pos, neg, y.mean(), tpr, fpr, fnr, ppv])

    return (pd.DataFrame(rows, columns=[group_col,"n","pos","neg","base_rate","TPR","FPR","FNR","PPV"])
              .sort_values("n", ascending=False))

# וידוא שיש topic_cols
topic_cols = [c for c in df_all.columns if c.startswith("topic_")]
assert len(topic_cols) > 0, "topic_cols not found. Run LDA first."

# Model A: base
pA, aucA = fit_predict(train, test, BASE_FEATURES)
# Model B: base + topics
pB, aucB = fit_predict(train, test, BASE_FEATURES + topic_cols)

print("AUC base:", round(aucA,4))
print("AUC base+topics:", round(aucB,4))

thr = 0.5

testA = test.copy()
testA["pred_y"] = (pA >= thr).astype(int)
fairA = fair_table_by_group(testA)

testB = test.copy()
testB["pred_y"] = (pB >= thr).astype(int)
fairB = fair_table_by_group(testB)

cmp = fairA.merge(fairB, on=GROUP_COL, suffixes=("_base","_topics"))
cmp["FPR_diff"] = cmp["FPR_topics"] - cmp["FPR_base"]
cmp["TPR_diff"] = cmp["TPR_topics"] - cmp["TPR_base"]

# תצוגה של הקבוצות הגדולות
cmp.sort_values("n_base", ascending=False).head(25)[
    [GROUP_COL,"n_base","base_rate_base",
     "TPR_base","TPR_topics","TPR_diff",
     "FPR_base","FPR_topics","FPR_diff"]
]


AUC base: 0.9064
AUC base+topics: 0.9061


,zip3,n_base,base_rate_base,TPR_base,TPR_topics,TPR_diff,FPR_base,FPR_topics,FPR_diff
0,945,474,0.111814,0.415094,0.433962,0.018868,0.064133,0.061758,-0.002375
1,750,459,0.165577,0.276316,0.276316,0.000000,0.033943,0.036554,0.002611
2,112,404,0.207921,0.380952,0.369048,-0.011905,0.071875,0.075000,0.003125
3,900,340,0.147059,0.460000,0.480000,0.020000,0.044828,0.048276,0.003448
4,331,319,0.188088,0.416667,0.400000,-0.016667,0.084942,0.092664,0.007722
5,917,310,0.161290,0.520000,0.520000,0.000000,0.061538,0.061538,0.000000
6,891,302,0.172185,0.403846,0.423077,0.019231,0.088000,0.084000,-0.004000


In [34]:
print(cmp["FPR_diff"].describe())
print(cmp["TPR_diff"].describe())


count    7.000000
mean     0.001504
std      0.003956
min     -0.004000
25%     -0.001188
50%      0.002611
75%      0.003287
max      0.007722
Name: FPR_diff, dtype: float64
count    7.000000
mean     0.004218
std      0.015388
min     -0.016667
25%     -0.005952
50%      0.000000
75%      0.019049
max      0.020000
Name: TPR_diff, dtype: float64


In [35]:
cmp.sort_values("FPR_diff", ascending=False).head(10)[[GROUP_COL,"n_base","FPR_base","FPR_topics","FPR_diff"]]
cmp.sort_values("FPR_diff", ascending=True).head(10)[[GROUP_COL,"n_base","FPR_base","FPR_topics","FPR_diff"]]


,zip3,n_base,FPR_base,FPR_topics,FPR_diff
6,891,302,0.088000,0.084000,-0.004000
0,945,474,0.064133,0.061758,-0.002375
5,917,310,0.061538,0.061538,0.000000
1,750,459,0.033943,0.036554,0.002611
2,112,404,0.071875,0.075000,0.003125
3,900,340,0.044828,0.048276,0.003448
4,331,319,0.084942,0.092664,0.007722


In [36]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

TARGET_COL = "is_default"
GROUP_COL = "zip3"

Xtr = train[BASE_FEATURES].fillna(0)
ytr = train[TARGET_COL].astype(int).values
Xte = test[BASE_FEATURES].fillna(0)
yte = test[TARGET_COL].astype(int).values

# משקל למחלקה חיובית (אופציונלי, מומלץ אם יש חוסר איזון)
pos = (ytr == 1).sum()
neg = (ytr == 0).sum()
scale_pos_weight = neg / max(1, pos)

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=1,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=123,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

xgb.fit(Xtr, ytr)
p = xgb.predict_proba(Xte)[:, 1]
auc = roc_auc_score(yte, p)
print("XGBoost AUC:", round(auc, 4))

test_xgb = test.copy()
test_xgb["pred_p"] = p
test_xgb["pred_y"] = (p >= 0.5).astype(int)


XGBoost AUC: 0.914


In [37]:
def fairness_gaps_topk(df, group_col="zip3", y_col="is_default", yhat_col="pred_y", k=30):
    top = df[group_col].value_counts().head(k).index
    d = df[df[group_col].isin(top)].copy()

    # metrics per group
    rows = []
    for g, x in d.groupby(group_col):
        y = x[y_col].values
        yhat = x[yhat_col].values
        pos = (y==1).sum()
        neg = (y==0).sum()
        tp = ((yhat==1) & (y==1)).sum()
        fp = ((yhat==1) & (y==0)).sum()

        tpr = tp / max(1, pos)
        fpr = fp / max(1, neg)
        pr  = yhat.mean()  # demographic parity (positive prediction rate)

        rows.append([g, len(x), y.mean(), tpr, fpr, pr])

    out = pd.DataFrame(rows, columns=[group_col,"n","base_rate","TPR","FPR","PPR"]).sort_values("n", ascending=False)

    # reference = overall across top-k
    ref = out[["TPR","FPR","PPR"]].mean()

    out["dTPR"] = out["TPR"] - ref["TPR"]
    out["dFPR"] = out["FPR"] - ref["FPR"]
    out["dPPR"] = out["PPR"] - ref["PPR"]
    out["EO_gap"] = np.maximum(out["dTPR"].abs(), out["dFPR"].abs())  # Equalized Odds gap proxy

    return out, ref

fair_xgb, ref_xgb = fairness_gaps_topk(test_xgb, k=30)
fair_xgb.head(20)


,zip3,n,base_rate,TPR,FPR,PPR,dTPR,dFPR,dPPR,EO_gap
26,945,474,0.111814,0.962264,0.223278,0.305907,0.051707,0.021219,-0.000270,0.051707
13,750,459,0.165577,0.881579,0.214099,0.324619,-0.028979,0.012040,0.018442,0.028979
2,112,404,0.207921,0.857143,0.268750,0.391089,-0.053415,0.066691,0.084912,0.066691
12,606,393,0.119593,0.936170,0.196532,0.284987,0.025613,-0.005527,-0.021189,0.025613
6,300,362,0.129834,0.893617,0.203175,0.292818,-0.016940,0.001115,-0.013359,0.016940
18,900,340,0.147059,0.920000,0.203448,0.308824,0.009443,0.001389,0.002647,0.009443
0,100,324,0.120370,0.923077,0.171930,0.262346,0.012519,-0.030129,-0.043831,0.030129
8,331,319,0.188088,0.950000,0.254826,0.385580,0.039443,0.052767,0.079403,0.052767
20,917,310,0.161290,0.960000,0.215385,0.335484,0.049443,0.013325,0.029307,0.049443
17,891,302,0.172185,0.980769,0.224000,0.354305,0.070212,0.021941,0.048128,0.070212


In [39]:
# Logistic predictions on the SAME test set
test_logit = test.copy()
test_logit["pred_p"] = pA          # או p0 / p1 בהתאם למה שמות המשתנים אצלך
test_logit["pred_y"] = (test_logit["pred_p"] >= 0.5).astype(int)

auc_logit = aucA                   # או auc0 / auc1 בהתאם


In [40]:
pA, aucA = fit_predict(train, test, BASE_FEATURES)

test_logit = test.copy()
test_logit["pred_p"] = pA
test_logit["pred_y"] = (pA >= 0.5).astype(int)

auc_logit = aucA


In [42]:
auc_xgb = auc
test_xgb 
test_xgb = test.copy()
test_xgb["pred_p"] = p
test_xgb["pred_y"] = (p >= 0.5).astype(int)


In [43]:
fair_logit, ref_logit = fairness_gaps_topk(test_logit, k=30)
fair_xgb, ref_xgb     = fairness_gaps_topk(test_xgb,  k=30)

summary = pd.DataFrame({
    "model": ["logit", "xgb"],
    "AUC": [auc_logit, auc_xgb],
    "EO_gap_mean": [fair_logit["EO_gap"].mean(), fair_xgb["EO_gap"].mean()],
    "EO_gap_max":  [fair_logit["EO_gap"].max(),  fair_xgb["EO_gap"].max()],
    "DP_gap_mean_abs": [fair_logit["dPPR"].abs().mean(), fair_xgb["dPPR"].abs().mean()],
})

summary


,model,AUC,EO_gap_mean,EO_gap_max,DP_gap_mean_abs
0,logit,0.906373,0.073472,0.260000,0.022403
1,xgb,0.914015,0.048757,0.160557,0.038840


In [44]:
fair_logit.sort_values("EO_gap", ascending=False).head(10)[["zip3","n","base_rate","EO_gap","dTPR","dFPR","dPPR"]]
fair_xgb.sort_values("EO_gap", ascending=False).head(10)[["zip3","n","base_rate","EO_gap","dTPR","dFPR","dPPR"]]


,zip3,n,base_rate,EO_gap,dTPR,dFPR,dPPR
15,840,198,0.141414,0.160557,-0.160557,-0.084412,-0.099106
3,113,219,0.109589,0.105633,0.006109,0.105633,0.068252
14,770,291,0.130584,0.089443,0.089443,-0.012336,-0.010644
5,201,211,0.085308,0.077224,-0.077224,-0.010349,-0.059731
24,928,199,0.120603,0.077224,-0.077224,-0.002059,-0.029795
17,891,302,0.172185,0.070212,0.070212,0.021941,0.048128
2,112,404,0.207921,0.066691,-0.053415,0.066691,0.084912
29,980,228,0.157895,0.061434,0.006109,-0.061434,-0.043019
28,956,199,0.145729,0.054960,0.054960,-0.043236,-0.029795
16,852,219,0.132420,0.054691,0.020477,-0.054691,-0.055035


In [45]:
fair_logit.sort_values("EO_gap", ascending=False).head(3)


,zip3,n,base_rate,TPR,FPR,PPR,dTPR,dFPR,dPPR,EO_gap
29,980,228,0.157895,0.666667,0.036458,0.135965,0.260000,-0.012735,0.034029,0.260000
28,956,199,0.145729,0.586207,0.029412,0.110553,0.179540,-0.019781,0.008617,0.179540
13,750,459,0.165577,0.276316,0.033943,0.074074,-0.130351,-0.015251,-0.027862,0.130351


טבלה לקבוצה

In [46]:
# נניח שיש לך fair_xgb ו-fair_logit
zip_summary = fair_xgb.merge(
    fair_logit[["zip3","EO_gap","dPPR"]].rename(columns={"EO_gap":"EO_gap_logit","dPPR":"dPPR_logit"}),
    on="zip3",
    how="left"
)

zip_summary = zip_summary.rename(columns={
    "EO_gap":"EO_gap_xgb",
    "dPPR":"dPPR_xgb"
})

zip_summary["EO_gap_delta"] = zip_summary["EO_gap_xgb"] - zip_summary["EO_gap_logit"]
zip_summary["dPPR_abs_xgb"] = zip_summary["dPPR_xgb"].abs()

zip_summary.sort_values("EO_gap_xgb", ascending=False).head(10)


,zip3,n,base_rate,TPR,FPR,PPR,dTPR,dFPR,dPPR_xgb,EO_gap_xgb,EO_gap_logit,dPPR_logit,EO_gap_delta,dPPR_abs_xgb
29,840,198,0.141414,0.750000,0.117647,0.207071,-0.160557,-0.084412,-0.099106,0.160557,0.049524,-0.041330,0.111034,0.099106
21,113,219,0.109589,0.916667,0.307692,0.374429,0.006109,0.105633,0.068252,0.105633,0.115000,-0.006046,-0.009367,0.068252
10,770,291,0.130584,1.000000,0.189723,0.295533,0.089443,-0.012336,-0.010644,0.089443,0.064561,-0.033207,0.024881,0.010644
24,201,211,0.085308,0.833333,0.191710,0.246445,-0.077224,-0.010349,-0.059731,0.077224,0.073333,-0.049803,0.003891,0.059731
27,928,199,0.120603,0.833333,0.200000,0.276382,-0.077224,-0.002059,-0.029795,0.077224,0.093333,0.008617,-0.016109,0.029795
9,891,302,0.172185,0.980769,0.224000,0.354305,0.070212,0.021941,0.048128,0.070212,0.038807,0.040448,0.031405,0.048128
2,112,404,0.207921,0.857143,0.268750,0.391089,-0.053415,0.066691,0.084912,0.066691,0.025714,0.034203,0.040977,0.084912
19,980,228,0.157895,0.916667,0.140625,0.263158,0.006109,-0.061434,-0.043019,0.061434,0.260000,0.034029,-0.198566,0.043019
28,956,199,0.145729,0.965517,0.158824,0.276382,0.054960,-0.043236,-0.029795,0.054960,0.179540,0.008617,-0.124581,0.029795
20,852,219,0.132420,0.931034,0.147368,0.251142,0.020477,-0.054691,-0.055035,0.054691,0.096322,-0.028877,-0.041631,0.055035


In [47]:
zip_summary["zip3"] = zip_summary["zip3"].astype(str).str.zfill(3)


In [48]:
import pandas as pd

census_path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\census_final_data\zip3_census_panel_2011_2014.csv"
census = pd.read_csv(census_path, low_memory=False)

print("Census shape:", census.shape)
print("Columns sample:", census.columns[:30].tolist())

# למצוא עמודת zip3 בקובץ
zip_candidates = [c for c in census.columns if c.lower() in {"zip3","zip","zipcode","zip_code","zcta3"}]
print("Zip candidates:", zip_candidates)

census.head()


Census shape: (3576, 12)
Columns sample: ['year', 'zip3', 'total_pop', 'black_nh', 'hispanic', 'poverty_count', 'poverty_universe', 'total_households', 'median_income_approx', 'share_black', 'share_hispanic', 'poverty_rate']
Zip candidates: ['zip3']


,year,zip3,total_pop,black_nh,hispanic,poverty_count,poverty_universe,total_households,median_income_approx,share_black,share_hispanic,poverty_rate
0,2011,6,1219302.0,859.0,1201175.0,633746.0,1208982.0,388337.0,15635.996907,0.000705,0.985133,0.524198
1,2011,7,1372589.0,1037.0,1363148.0,640366.0,1358346.0,436994.0,18646.845288,0.000756,0.993122,0.471431
2,2011,9,1148444.0,2386.0,1131444.0,401066.0,1131250.0,404159.0,26922.547912,0.002078,0.985197,0.354533
3,2011,10,468019.0,9265.0,43435.0,51737.0,436107.0,181490.0,57738.417419,0.019796,0.092806,0.118634
4,2011,11,168764.0,30760.0,57868.0,39898.0,161233.0,61816.0,43162.452812,0.182266,0.342893,0.247456


In [49]:
# 2011 snapshot (מומלץ)
census_2011 = census[census["year"] == 2011].copy()

# לוודא zip3 בפורמט תואם בשני הצדדים
zip_summary["zip3"] = zip_summary["zip3"].astype(str).str.zfill(3)
census_2011["zip3"] = census_2011["zip3"].astype(str).str.zfill(3)

print("census_2011:", census_2011.shape)


census_2011: (894, 12)


In [50]:
zip_enriched = zip_summary.merge(census_2011, on="zip3", how="left")

print("zip_enriched shape:", zip_enriched.shape)

# בדיקת חסרים (אם יש zip3 שלא נמצא במפקד)
miss = zip_enriched["median_income_approx"].isna().mean()
print("Missing census share:", miss)

zip_enriched.head(10)


zip_enriched shape: (30, 25)
Missing census share: 0.0


,zip3,n,base_rate,TPR,FPR,PPR,dTPR,dFPR,dPPR_xgb,EO_gap_xgb,...,total_pop,black_nh,hispanic,poverty_count,poverty_universe,total_households,median_income_approx,share_black,share_hispanic,poverty_rate
0,945,474,0.111814,0.962264,0.223278,0.305907,0.051707,0.021219,-0.000270,0.051707,...,2249353.0,170440.0,512561.0,197578.0,2217737.0,781652.0,83694.066119,0.075773,0.227870,0.089090
1,750,459,0.165577,0.881579,0.214099,0.324619,-0.028979,0.012040,0.018442,0.028979,...,2041979.0,199008.0,481404.0,195396.0,2029583.0,720046.0,73499.807655,0.097458,0.235754,0.096274
2,112,404,0.207921,0.857143,0.268750,0.391089,-0.053415,0.066691,0.084912,0.066691,...,2486119.0,807108.0,492496.0,545963.0,2465053.0,907785.0,47055.280649,0.324646,0.198098,0.221481
3,606,393,0.119593,0.936170,0.196532,0.284987,0.025613,-0.005527,-0.021189,0.025613,...,2686081.0,884831.0,754494.0,562925.0,2638713.0,1025452.0,50371.435581,0.329413,0.280890,0.213333
4,300,362,0.129834,0.893617,0.203175,0.292818,-0.016940,0.001115,-0.013359,0.016940,...,2163748.0,658951.0,261991.0,261523.0,2139449.0,765221.0,66244.417666,0.304541,0.121082,0.122238
5,900,340,0.147059,0.920000,0.203448,0.308824,0.009443,0.001389,0.002647,0.009443,...,2393455.0,332569.0,1261616.0,570465.0,2337262.0,836410.0,47346.735020,0.138949,0.527111,0.244074
6,100,324,0.120370,0.923077,0.171930,0.262346,0.012519,-0.030129,-0.043831,0.030129,...,1506425.0,203676.0,393798.0,264759.0,1461442.0,692251.0,72886.569712,0.135205,0.261412,0.181163
7,331,319,0.188088,0.950000,0.254826,0.385580,0.039443,0.052767,0.079403,0.052767,...,1862499.0,324834.0,1152827.0,306749.0,1827743.0,641289.0,49374.691966,0.174408,0.618968,0.167829
8,917,310,0.161290,0.960000,0.215385,0.335484,0.049443,0.013325,0.029307,0.049443,...,1927494.0,73655.0,967619.0,219046.0,1890686.0,551528.0,66440.375629,0.038213,0.502009,0.115855
9,891,302,0.172185,0.980769,0.224000,0.354305,0.070212,0.021941,0.048128,0.070212,...,1402815.0,135547.0,425456.0,189956.0,1388433.0,515138.0,55772.170208,0.096625,0.303287,0.136813


In [51]:
zip_enriched["poverty_rate_calc"] = zip_enriched["poverty_count"] / zip_enriched["poverty_universe"]
zip_enriched["poverty_rate_calc"] = zip_enriched["poverty_rate_calc"].clip(0, 1)

# sanity check מול poverty_rate (אמור להיות דומה)
print(zip_enriched[["poverty_rate", "poverty_rate_calc"]].corr())


                   poverty_rate  poverty_rate_calc
poverty_rate                1.0                1.0
poverty_rate_calc           1.0                1.0


In [52]:
EO_COL = "EO_gap_xgb" if "EO_gap_xgb" in zip_enriched.columns else "EO_gap"

cols = ["median_income_approx", "poverty_rate", "share_black", "share_hispanic", "total_pop", "total_households"]

corr = zip_enriched[[EO_COL] + cols].corr(numeric_only=True)[EO_COL].sort_values(ascending=False)
print(corr)


EO_gap_xgb              1.000000
median_income_approx    0.086703
share_hispanic         -0.122447
total_pop              -0.157830
poverty_rate           -0.189715
total_households       -0.210507
share_black            -0.211020
Name: EO_gap_xgb, dtype: float64


In [53]:
view_cols = [
    "zip3", "n", "base_rate", EO_COL,
    "TPR", "FPR", "PPR",
    "median_income_approx", "poverty_rate", "share_black", "share_hispanic",
    "total_pop", "total_households"
]

zip_enriched.sort_values(EO_COL, ascending=False)[view_cols].head(15)


,zip3,n,base_rate,EO_gap_xgb,TPR,FPR,PPR,median_income_approx,poverty_rate,share_black,share_hispanic,total_pop,total_households
29,840,198,0.141414,0.160557,0.750000,0.117647,0.207071,69889.125345,0.076497,0.007468,0.099202,1209871.0,365111.0
21,113,219,0.109589,0.105633,0.916667,0.307692,0.374429,57091.164842,0.138087,0.041586,0.315658,1148768.0,406825.0
10,770,291,0.130584,0.089443,1.000000,0.189723,0.295533,52392.399920,0.199613,0.222465,0.435269,2873899.0,1012847.0
24,201,211,0.085308,0.077224,0.833333,0.191710,0.246445,111585.922260,0.047632,0.082684,0.143773,820954.0,277117.0
27,928,199,0.120603,0.077224,0.833333,0.200000,0.276382,74917.322344,0.113831,0.026857,0.404904,1222665.0,365550.0
9,891,302,0.172185,0.070212,0.980769,0.224000,0.354305,55772.170208,0.136813,0.096625,0.303287,1402815.0,515138.0
2,112,404,0.207921,0.066691,0.857143,0.268750,0.391089,47055.280649,0.221481,0.324646,0.198098,2486119.0,907785.0
19,980,228,0.157895,0.061434,0.916667,0.140625,0.263158,81152.673829,0.082085,0.038860,0.085614,1328070.0,516757.0
28,956,199,0.145729,0.054960,0.965517,0.158824,0.276382,66186.729547,0.109884,0.039798,0.177642,1160348.0,424577.0
20,852,219,0.132420,0.054691,0.931034,0.147368,0.251142,65191.637353,0.110421,0.031584,0.187536,1425804.0,552101.0


מתאם OE_GAP מול N

In [54]:
zip_enriched["n"] = zip_enriched["n"].astype(float)
zip_enriched[[EO_COL, "n", "total_pop", "total_households"]].corr(numeric_only=True)[EO_COL]


EO_gap_xgb          1.000000
n                  -0.226150
total_pop          -0.157830
total_households   -0.210507
Name: EO_gap_xgb, dtype: float64

In [57]:
fair_xgb_big = fair_xgb[fair_xgb["n"] >= 300].copy()
fair_logit_big = fair_logit[fair_logit["n"] >= 300].copy()

print("xgb groups:", fair_xgb_big.shape[0], " | logit groups:", fair_logit_big.shape[0])
print("EO mean xgb:", fair_xgb_big["EO_gap"].mean(), "EO max xgb:", fair_xgb_big["EO_gap"].max())
print("EO mean logit:", fair_logit_big["EO_gap"].mean(), "EO max logit:", fair_logit_big["EO_gap"].max())


xgb groups: 10  | logit groups: 10
EO mean xgb: 0.04019223434531599 EO max xgb: 0.07021175057593687
EO mean logit: 0.05478411601784263 EO max logit: 0.13035074971336352


In [58]:
# על test_xgb, בנה bins של zip3 לפי n ב-test
counts = test_xgb["zip3"].value_counts().rename("n_zip").reset_index().rename(columns={"index":"zip3"})
tmp = test_xgb.merge(counts, on="zip3", how="left")

tmp["zip_size_bin"] = pd.qcut(tmp["n_zip"], q=5, labels=[1,2,3,4,5])

# fairness לפי bin במקום zip3
def fair_by_bin(df, bin_col="zip_size_bin", y_col="is_default", yhat_col="pred_y"):
    rows = []
    for g, x in df.groupby(bin_col):
        y = x[y_col].values
        yhat = x[yhat_col].values
        pos = (y==1).sum(); neg = (y==0).sum()
        tp = ((yhat==1)&(y==1)).sum()
        fp = ((yhat==1)&(y==0)).sum()
        tpr = tp/max(1,pos); fpr = fp/max(1,neg); ppr = yhat.mean()
        rows.append([g, len(x), y.mean(), tpr, fpr, ppr])
    return pd.DataFrame(rows, columns=[bin_col,"n","base_rate","TPR","FPR","PPR"]).sort_values("n", ascending=False)

fair_bins = fair_by_bin(tmp)
fair_bins


C:\Users\ariel\AppData\Local\Temp\ipykernel_39796\1773979745.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for g, x in df.groupby(bin_col):


,zip_size_bin,n,base_rate,TPR,FPR,PPR
1,2,7562,0.156308,0.906937,0.213323,0.321740
3,4,7500,0.146000,0.893151,0.189227,0.292000
0,1,7480,0.160829,0.903574,0.211407,0.322727
4,5,7279,0.148509,0.918594,0.209906,0.315153
2,3,7252,0.154302,0.910634,0.197130,0.307226


טבלת השוואה סופית

In [59]:
summary_big = pd.DataFrame({
    "model": ["logit","xgb"],
    "groups(n>=300)": [fair_logit_big.shape[0], fair_xgb_big.shape[0]],
    "EO_gap_mean": [fair_logit_big["EO_gap"].mean(), fair_xgb_big["EO_gap"].mean()],
    "EO_gap_max":  [fair_logit_big["EO_gap"].max(),  fair_xgb_big["EO_gap"].max()],
})
summary_big


,model,groups(n>=300),EO_gap_mean,EO_gap_max
0,logit,10,0.054784,0.130351
1,xgb,10,0.040192,0.070212


In [60]:
fair_xgb_big.sort_values("EO_gap", ascending=False).head(10)[["zip3","n","base_rate","EO_gap","dTPR","dFPR","dPPR"]]
fair_logit_big.sort_values("EO_gap", ascending=False).head(10)[["zip3","n","base_rate","EO_gap","dTPR","dFPR","dPPR"]]


,zip3,n,base_rate,EO_gap,dTPR,dFPR,dPPR
13,750,459,0.165577,0.130351,-0.130351,-0.015251,-0.027862
20,917,310,0.161290,0.113333,0.113333,0.012345,0.033548
6,300,362,0.129834,0.061419,0.061419,0.007950,0.008561
18,900,340,0.147059,0.053333,0.053333,-0.004366,0.003946
12,606,393,0.119593,0.044964,-0.044964,-0.020291,-0.033234
17,891,302,0.172185,0.038807,-0.002820,0.038807,0.040448
8,331,319,0.188088,0.035749,0.010000,0.035749,0.045399
0,100,324,0.120370,0.029231,0.029231,-0.017614,-0.021689
2,112,404,0.207921,0.025714,-0.025714,0.022682,0.034203
26,945,474,0.111814,0.014940,0.008428,0.014940,0.001440
